# 整体架构流程图
![整体架构流程图](../image/gpt整体架构流程图.png)


# 1. 导入相关包

In [1]:
import torch
import torch.nn as nn
import math

# 2. 数据准备

In [2]:
sentences = {
    "我今天去公园", # 拆分成 ["我", "今","天", "去", "公","园"]
    "公园里有很多树", # 拆分成 ["公","园", "里", "有", "很","多", "树"]
    "树上有小鸟" # 拆分成 ["树", "上", "有", "小","鸟"]
}

# 构建有序词表（按首次出现顺序）
chars = [] # 初始化一个空列表
seen = set() # 初始化一个空集合,用于去重
for s in sentences: # 遍历每个句子
    for c in s: # 遍历句子中的每个字符
        if c not in seen: # 如果字符没有出现过
            seen.add(c) # 将字符添加到集合中
            chars.append(c) # 将字符添加到列表中,保持顺序

char2idx = {char:idx+2 for idx,char in enumerate(chars)} # 构建字符到索引的映射,索引从2开始,0和1保留给特殊标记
char2idx["<pad>"] = 0 # 添加填充标记
char2idx["<unk>"] = 1 # 添加未知标记
vocab_size = len(char2idx) # 计算词表大小
idx2char = {v:k for k,v in char2idx.items()} # 构建索引到字符的映射,方便后续解码

# 将句子转换为索引序列
def text_to_ids(text):
    return [char2idx.get(c,1) for c in text] # 将文本中的每个字符转换为对应的索引,如果字符不在词表中则使用<unk>的索引1

data = [text_to_ids(s) for s in sentences] # 将每个句子转换为索引序列

# 3. 位置编码

## 正弦/余弦位置编码

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self,d_model:int,max_len:int = 5000): # 初始化位置编码,d_model是词向量维度,max_len是最大序列长度
        super().__init__() # 调用父类的初始化方法
        pe = torch.zeros(max_len,d_model) # 创建一个全零的张量,形状为(max_len, d_model)
        position = torch.arange(0,max_len,dtype=torch.float).unsqueeze(1) # 创建一个位置索引的张量,形状为(max_len, 1)
        div_term = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model)) # 计算位置编码中的分母项,形状为(d_model//2)
        pe[:,0::2] = torch.sin(position*div_term) # 对偶数维度应用正弦函数
        pe[:,1::2] = torch.cos(position*div_term) # 对奇数维度应用余弦函数
        pe = pe.unsqueeze(0) # 在第0维添加一个批次维度,形状变为(1, max_len, d_model)
        self.register_buffer('pe',pe) # 将位置编码注册为模型的缓冲区,这样它不会被更新但会随模型一起保存

    def forward(self,x): # 前向传播方法,输入x的形状为(batch_size, seq_len, d_model)
        x = x+self.pe[:,:x.size(1),:] # 将输入与对应位置的编码相加,形状保持不变
        return x # 返回加上位置编码后的结果

# 4. 定义Transformer模型（微调参数）

In [4]:
class ChineseTransformer(nn.Module):
    def __init__(self,vocab_size:int,d_model:int=32,nhead:int=4,num_layers:int=2): # 初始化Transformer模型,vocab_size是词表大小,d_model是词向量维度,nhead是多头注意力的头数
        super().__init__() # 调用父类的初始化方法
        self.embed = nn.Embedding(vocab_size,d_model) # 定义词嵌入层,将输入的索引转换为词向量
        self.pos_enc = PositionalEncoding(d_model,max_len=20) # 定义位置编码层,最大序列长度设置为20
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, # 词向量维度
            nhead=nhead, # 多头注意力的头数
            dim_feedforward=128, # 前馈网络的隐藏层维度
            dropout=0.1, # dropout率
        )
        self.transformer = nn.TransformerEncoder(encoder_layer,num_layers=num_layers) # 定义Transformer编码器,由num_layers层组成
        self.fc = nn.Linear(d_model,vocab_size) # 定义输出层,将Transformer的输出映射到词表大小的维度

    def forward(self,x): # 前向传播方法,输入x的形状为(batch_size, seq_len)
        x = self.embed(x) # 将输入的索引转换为词向量,形状变为(batch_size, seq_len, d_model)
        x = self.pos_enc(x) # 添加位置编码,形状保持不变
        x = x.permute(1,0,2) # 调整维度顺序以适应Transformer的输入要求,形状变为(seq_len, batch_size, d_model)
        x = self.transformer(x) # 通过Transformer编码器处理输入,形状保持不变
        x = x.permute(1,0,2) # 调整维度顺序回原来的格式,形状变为(batch_size, seq_len, d_model)
        x = self.fc(x) # 通过输出层映射到词表大小的维度,形状变为(batch_size, seq_len, vocab_size)
        return x # 返回模型的输出

# 5. 数据预处理

In [5]:
def create_sequences(data,seq_length=3): # 创建输入输出序列,data是索引序列列表,seq_length是输入序列的长度
    inputs,targets = [],[] # 初始化输入和目标列表
    for seq in data: # 遍历每个索引序列
        for i in range(len(seq)-seq_length): # 遍历序列,生成输入输出对
            inputs.append(seq[i:i+seq_length]) # 输入是当前索引开始的seq_length长度的子序列
            targets.append(seq[i+1:i+1+seq_length]) # 目标是输入序列的下一个位置开始的seq_length长度的子序列
    return torch.tensor(inputs),torch.tensor(targets) # 将输入和目标列表转换为张量并返回

# 生成训练数据（序列长度设为3）
inputs,targets = create_sequences(data,seq_length=3) # 调用create_sequences函数生成输入和目标张量
print("示例输入-目标对：")
print("输入：",[idx2char[i.item()] for i in inputs[0]],"->目标：",[idx2char[t.item()] for t in targets[0]]) # 打印第一个输入-目标对的字符形式,方便理解

示例输入-目标对：
输入： ['我', '今', '天'] ->目标： ['今', '天', '去']


# 6. 训练模型

In [6]:
model = ChineseTransformer(vocab_size=vocab_size) # 创建Transformer模型实例
criterion = nn.CrossEntropyLoss(ignore_index=0) # 定义损失函数,忽略填充标记的索引0
optimizer = torch.optim.Adam(model.parameters(),lr=0.001) # 定义优化器,使用Adam算法,学习率为0.001

num_epochs = 100 # 设置训练的轮数
for epoch in range(num_epochs): 
    model.train() # 设置模型为训练模式
    optimizer.zero_grad() # 清除之前的梯度

    # 前向传播
    output = model(inputs) # 将输入数据传入模型,得到输出,形状为(batch_size, seq_len, vocab_size)
    # 计算损失
    loss = criterion(output.view(-1,vocab_size),targets.view(-1)) # 将输出和目标展平后计算交叉熵损失

    # 反向传播和优化
    loss.backward() # 反向传播计算梯度
    optimizer.step() # 更新模型参数

    # 每20轮打印一次损失
    if(epoch+1)%20==0:
        print(f"Epoch[{epoch+1}/{num_epochs}],Loss:{loss.item():.4f}") # 打印当前轮数和损失值

C:\Users\zjm\AppData\Local\Temp\ipykernel_30012\716752119.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(encoder_layer,num_layers=num_layers) # 定义Transformer编码器,由num_layers层组成


Epoch[20/100],Loss:1.2027
Epoch[40/100],Loss:0.5316
Epoch[60/100],Loss:0.2890
Epoch[80/100],Loss:0.1796
Epoch[100/100],Loss:0.1342


# 7.推理测试

In [8]:
def predict_next(text,model,temperature=1.0): # 定义一个函数用于预测下一个字符,text是输入文本,model是训练好的模型,temperature是控制生成文本多样性的参数
    model.eval() # 设置模型为评估模式
    with torch.no_grad(): # 在不计算梯度的上下文中执行
        # 将输入文本转换为索引
        input_ids = torch.tensor([char2idx.get(c,1) for c in text[-3:]],dtype=torch.long).unsqueeze(0) # 将输入文本的最后3个字符转换为索引,如果字符不在词表中则使用<unk>的索引1,并添加批次维度
        output = model(input_ids) # 将输入索引传入模型,得到输出,形状为(batch_size, seq_len, vocab_size)
        next_token_logits = output[0,-1,:]/temperature # 获取最后一个时间步的输出,并根据temperature调整生成文本的多样性
        probs = torch.softmax(next_token_logits,dim=-1) # 对调整后的输出应用softmax函数,得到每个字符的概率分布
        next_token_id = torch.argmax(probs).item() # 获取概率最高的字符索引
        return idx2char[next_token_id] # 返回预测的下一个字符
    
# 测试模型的推理能力
test_cases = ["我今天","公园里","树上有"] # 定义一些测试输入文本
for case in test_cases: # 遍历每个测试输入
    predicted = predict_next(case,model) # 调用predict_next函数预测下一个字符
    print(f"输入: '{case}' -> 预测下一个字符: '{predicted}'") # 打印输入文本和预测的下一个字符

输入: '我今天' -> 预测下一个字符: '去'
输入: '公园里' -> 预测下一个字符: '有'
输入: '树上有' -> 预测下一个字符: '小'
